In [1]:
# ------------------------------------------
# NOTEBOOK DE CONSOLIDAÇÃO DE FEATURE ENGINEERING
# ------------------------------------------

In [2]:
# ------------------------------------------
# TAXA DE CONVERSAO BRL/USD : 4.40
# ------------------------------------------
PTAX = 4.40

In [3]:
# ------------------------------------------
# modulos pip
# ------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [4]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
# ------------------------------------------
# DATABASE: historico dos couriers churneados pela plataforma
#
# SK.CREATED_AT::DATE : data de cadastro do courier na plataforma
# FECHA_ULT : ultima data de acesso do app pelo courier
#
# NOTA: esse database foi recebido com linhas duplicadas
# apos tratamento, o total de linhas caiu de 32 milhões para 152,2 mil
# ------------------------------------------
df_churn = pd.read_csv("/content/drive/Shareddrives/grupo4-rappi-hour/bases-rappi/criacao-contas-churn-sem-duplicados-prov.csv")
df_churn.drop(labels="Unnamed: 0", axis=1, inplace = True)
print(f"Total de linhas: {df_churn.shape[0]}")
df_churn.head(3)

Total de linhas: 152224


,ID,FIRST_NAME,GENDER,CITY,SK.CREATED_AT::DATE,TRANSPORT_MEDIA_TYPE,CARTAO,LEVEL_NAME,FECHA_ULT
0,1286316,Adailton,M,Grande São Paulo,2021-06-07,motorbike,True,bronze,2022-01-16T23:27:35Z
1,1110698,Adriano Floriano Da Silva,M,Recife,2021-02-11,bicycle,True,bronze,2021-07-15T11:16:04Z
2,284886,Bruno,M,Grande São Paulo,2019-07-03,motorbike,False,bronze,2021-07-07T12:33:21Z


In [6]:
# ------------------------------------------
# database contem dados de cidades estrangeiras e nulos
# EXEMPLO : Tuxtla Gutierrez [MX] ; without_city
# ------------------------------------------
df_churn[df_churn['CITY'].isin([ "Tuxtla Gutierrez", "without_city" ])].head()

,ID,FIRST_NAME,GENDER,CITY,SK.CREATED_AT::DATE,TRANSPORT_MEDIA_TYPE,CARTAO,LEVEL_NAME,FECHA_ULT
8727,429842,Marcilon Antônio,M,without_city,2019-10-19,bicycle,True,rookie,2021-09-09T13:48:10Z
10994,1310230,Carla,F,without_city,2021-07-02,motorbike,False,bronze,2021-08-30T15:33:51Z
11318,1127727,Leandro,M,without_city,2021-02-21,bicycle,True,bronze,2021-09-09T13:04:48Z
12060,584838,Wadson,M,Tuxtla Gutierrez,2020-02-27,motorbike,True,bronze,2021-08-06T22:54:12Z
21241,547765,Thyago,M,without_city,2020-01-27,motorbike,True,bronze,2021-09-09T17:50:33Z


In [7]:
# ------------------------------------------
# database contem couriers que abandonaram a plataforma
# entre junho de 2021 a julho de 2022
# ------------------------------------------
df_churn["FECHA_ULT"].map(lambda x : x[:7]).value_counts().sort_index()

2021-06    11633
2021-07    12387
2021-08    11777
2021-09     9838
2021-10     9760
2021-11     8653
2021-12     8374
2022-01     9337
2022-02     9966
2022-03    13781
2022-04    13602
2022-05    15140
2022-06    16030
2022-07     1946
Name: FECHA_ULT, dtype: int64

In [8]:
# ------------------------------------------
# 80% do churn está concentrado nas seguintes cidades
#
# DECISAO : trabalhar apenas com couriers que atuem nessas cidades
# ------------------------------------------
df_conc_churn = (df_churn["CITY"].value_counts(normalize=True).cumsum() < 0.8).to_frame()
df_conc_by_city = df_conc_churn[df_conc_churn["CITY"] == True]
df_conc_by_city.index

Index(['Grande São Paulo', 'Rio de Janeiro', 'Belo Horizonte', 'Recife',
       'Curitiba', 'Fortaleza', 'Porto Alegre'],
      dtype='object')

In [9]:
# ------------------------------------------
# CONCLUSAO - FEATURE ENGINEERING - DATABASES:
#     . CRIACAO CONTAS CHURN : filtragem dos couriers que atuam nos principais
#       mercados
# ------------------------------------------
filter = df_conc_by_city.index.to_list()
df_churn_conc_by_city = df_churn[df_churn['CITY'].isin(filter)]
print(f"Total de linhas: {df_churn_conc_by_city.shape[0]}")
df_churn_conc_by_city.head(3)

Total de linhas: 120206


,ID,FIRST_NAME,GENDER,CITY,SK.CREATED_AT::DATE,TRANSPORT_MEDIA_TYPE,CARTAO,LEVEL_NAME,FECHA_ULT
0,1286316,Adailton,M,Grande São Paulo,2021-06-07,motorbike,True,bronze,2022-01-16T23:27:35Z
1,1110698,Adriano Floriano Da Silva,M,Recife,2021-02-11,bicycle,True,bronze,2021-07-15T11:16:04Z
2,284886,Bruno,M,Grande São Paulo,2019-07-03,motorbike,False,bronze,2021-07-07T12:33:21Z


In [10]:
# ------------------------------------------
# DATABASE : informacoes gerais de couriers ativos na plataforma
#
# is_active (bool) : true se ativo ; false se inativo
# ------------------------------------------
df_infos_gerais = pd.read_csv("/content/drive/Shareddrives/grupo4-rappi-hour/bases-rappi/infos-gerais-prov.csv")
print(f"Total de linhas: {df_infos_gerais.shape[0]}")
perc_ativos = round(df_infos_gerais["IS_ACTIVE"].value_counts(normalize=True)[1] * 100, 1)
print(f"% de couriers ativos no database: {perc_ativos}%")
df_infos_gerais.head(3)

Total de linhas: 180178
% de couriers ativos no database: 99.8%


,ID,NOME,SOBRENOME,GENERO,DATA_NASCIMENTO,CIDADE,IS_ACTIVE,TRANSPORTE,AUTO_ACEITE,COUNT_ORDERS_LAST_7D,...,ULTIMO_PEDIDO,COUNT_ORDERS_RESTAURANTES,COUNT_ORDERS_MERCADO,COUNT_ORDERS_FARMACIA,COUNT_ORDERS_EXPRESS,COUNT_ORDERS_ECOMMERCE,COUNT_ORDERS_ANTOJO,FRETE_MEDIO,COOKING_TIME_MEDIO,ITENS_MEDIO
0,1561246,Wilton Jhonne,Da Silva Abreu,M,1988-04-21,Sao Paulo,True,motorbike,True,1,...,2022-08-01T00:00:00Z,1,0,0,0,0,0,62.255500,10.000000,1.0
1,1561210,Dennis Leonardo,Pereira Santos,M,1998-06-28,Grande São Paulo,True,motorbike,True,7,...,2022-08-01T00:00:00Z,6,1,0,0,0,0,43.444714,23.142857,3.0
2,1561205,Thavillo Teles,Balviano De Olivetra,M,1997-09-25,Natal,True,motorbike,True,2,...,2022-08-01T00:00:00Z,1,1,0,0,0,0,42.230500,4.500000,3.5


In [11]:
# ------------------------------------------
# padronizacao de grafia das cidades usadas para
# filtragem dos couriers
# ------------------------------------------
df_infos_gerais.replace(
    {"CIDADE": {
        'Sao Paulo': 'Grande São Paulo',
        'São Paulo': 'Grande São Paulo',
        'SÃO PAULO': 'Grande São Paulo',
        'BELO HORIZINTE': 'Belo Horizonte',
        'RIO DE JANEIRO': 'Rio de Janeiro'
    }}, inplace=True
)

In [12]:
# ------------------------------------------
# filtragem dos couriers ativos que atuam nos principais mercados 
# ------------------------------------------
df_infos_gerais_conc_by_city = df_infos_gerais[df_infos_gerais['CIDADE'].isin(filter)]
df_infos_gerais_conc_by_city["CIDADE"].unique()

array(['Grande São Paulo', 'Recife', 'Fortaleza', 'Belo Horizonte',
       'Rio de Janeiro', 'Porto Alegre', 'Curitiba'], dtype=object)

In [13]:
# ------------------------------------------
# cruzamento dos couriers inativos de acordo com o database de churn
# e os dados gerais de couriers ativos do database de infos gerais
# 
# RESULTADO ESPERADO : baixa repeticao de IDs entre os databases
# RESULTADO OBSERVADO : alta repeticao de IDs (101.140)
#
# CONCLUSAO : repeticao elevada reflete couriers que abandonam a
# plataforma periodicamente e acabam retornando apos um periodo
# ------------------------------------------
filter = df_churn_conc_by_city["ID"].to_list()
df_infos_gerais_conc_by_city[df_infos_gerais_conc_by_city["ID"].isin(filter)].shape[0]

101140

In [14]:
# ------------------------------------------
# EXEMPLO (part 1 de 2):
# id 1544047 - Raquel da Costa Mendes, consta como churn, tendo
# acessado a plataforma pela ultima vez em 03-07-2022
# ------------------------------------------
print(df_churn_conc_by_city[df_churn_conc_by_city["ID"] == 1544047])

            ID FIRST_NAME GENDER            CITY SK.CREATED_AT::DATE  \
44151  1544047     Raquel      F  Belo Horizonte          2022-07-03   

      TRANSPORT_MEDIA_TYPE CARTAO LEVEL_NAME             FECHA_ULT  
44151                  car  False     rookie  2022-07-03T20:29:39Z  


In [15]:
# ------------------------------------------
# EXEMPLO (part 2 de 2):
# id 1544047 - Raquel da Costa Mendes, consta com status ATIVO
# ------------------------------------------
print(df_infos_gerais_conc_by_city[df_infos_gerais_conc_by_city["ID"] == 1544047])

           ID    NOME        SOBRENOME GENERO DATA_NASCIMENTO          CIDADE  \
3572  1544047  Raquel  Da Costa Mendes      F      1983-03-01  Belo Horizonte   

      IS_ACTIVE TRANSPORTE  AUTO_ACEITE  COUNT_ORDERS_LAST_7D  ...  \
3572       True        car         True                     0  ...   

             ULTIMO_PEDIDO  COUNT_ORDERS_RESTAURANTES  COUNT_ORDERS_MERCADO  \
3572  2022-07-03T00:00:00Z                          0                     0   

      COUNT_ORDERS_FARMACIA COUNT_ORDERS_EXPRESS COUNT_ORDERS_ECOMMERCE  \
3572                      0                    0                      0   

      COUNT_ORDERS_ANTOJO  FRETE_MEDIO  COOKING_TIME_MEDIO  ITENS_MEDIO  
3572                    0      53.3555                 NaN          2.0  

[1 rows x 25 columns]


In [16]:
# ------------------------------------------
# CONCLUSAO - FEATURE ENGINEERING - DATABASES:
#     . CRIACAO CONTAS CHURN
#     . INFOS GERAIS 
#
# DECISAO : tierizar os dados de churn (col: IS_ACTIVE)
#     . categoria 1 : True - couriers que nunca foram churneados
#     . categoria 2 : Quasi - couriers inconsistentes
#     . categoria 3 : False - couriers com churn definitivo
# ------------------------------------------
df_infos_gerais_conc_by_city.head(2)

,ID,NOME,SOBRENOME,GENERO,DATA_NASCIMENTO,CIDADE,IS_ACTIVE,TRANSPORTE,AUTO_ACEITE,COUNT_ORDERS_LAST_7D,...,ULTIMO_PEDIDO,COUNT_ORDERS_RESTAURANTES,COUNT_ORDERS_MERCADO,COUNT_ORDERS_FARMACIA,COUNT_ORDERS_EXPRESS,COUNT_ORDERS_ECOMMERCE,COUNT_ORDERS_ANTOJO,FRETE_MEDIO,COOKING_TIME_MEDIO,ITENS_MEDIO
0,1561246,Wilton Jhonne,Da Silva Abreu,M,1988-04-21,Grande São Paulo,True,motorbike,True,1,...,2022-08-01T00:00:00Z,1,0,0,0,0,0,62.255500,10.000000,1.0
1,1561210,Dennis Leonardo,Pereira Santos,M,1998-06-28,Grande São Paulo,True,motorbike,True,7,...,2022-08-01T00:00:00Z,6,1,0,0,0,0,43.444714,23.142857,3.0


In [17]:
# ------------------------------------------
# filter : lista dos couriers flagados como churn no database CRIACAO CONTAS CHURN
# mask : checa se os couriers flagados constam como ativos no database INFOS GERAIS
# CONCLUSAO : se houver match, o courier é inconsistente e categorizado como Quasi
# ------------------------------------------
filter = df_churn_conc_by_city["ID"].to_list()
mask = df_infos_gerais_conc_by_city["ID"].isin(filter)
column_name = "IS_ACTIVE"
df_infos_gerais_conc_by_city.loc[mask, column_name] = "Quasi"

/usr/local/lib/python3.7/dist-packages/pandas/core/indexing.py:1817: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_column(loc, value, pi)


In [18]:
# ------------------------------------------
# checagem da distribuicao de categorias apos aplicacao do filtro
# ------------------------------------------
df_infos_gerais_conc_by_city["IS_ACTIVE"].value_counts(normalize=True)

Quasi    0.719806
True     0.279311
False    0.000882
Name: IS_ACTIVE, dtype: float64

In [19]:
# ------------------------------------------
# MERGE DOS DATABASES : CRIACAO CONTAS CHURN *AND* INFOS GERAIS
# OBJETIVO : incluir couriers com churn definitivo no database de consolidacao
# ------------------------------------------

In [20]:
print(f"Dimensões do database de churn: {df_churn_conc_by_city.shape}")
df_churn_conc_by_city.head(2)

Dimensões do database de churn: (120206, 9)


,ID,FIRST_NAME,GENDER,CITY,SK.CREATED_AT::DATE,TRANSPORT_MEDIA_TYPE,CARTAO,LEVEL_NAME,FECHA_ULT
0,1286316,Adailton,M,Grande São Paulo,2021-06-07,motorbike,True,bronze,2022-01-16T23:27:35Z
1,1110698,Adriano Floriano Da Silva,M,Recife,2021-02-11,bicycle,True,bronze,2021-07-15T11:16:04Z


In [21]:
print(f"Dimensões do database de churn: {df_infos_gerais_conc_by_city.shape}")
df_infos_gerais_conc_by_city.head(2)

Dimensões do database de churn: (140510, 25)


,ID,NOME,SOBRENOME,GENERO,DATA_NASCIMENTO,CIDADE,IS_ACTIVE,TRANSPORTE,AUTO_ACEITE,COUNT_ORDERS_LAST_7D,...,ULTIMO_PEDIDO,COUNT_ORDERS_RESTAURANTES,COUNT_ORDERS_MERCADO,COUNT_ORDERS_FARMACIA,COUNT_ORDERS_EXPRESS,COUNT_ORDERS_ECOMMERCE,COUNT_ORDERS_ANTOJO,FRETE_MEDIO,COOKING_TIME_MEDIO,ITENS_MEDIO
0,1561246,Wilton Jhonne,Da Silva Abreu,M,1988-04-21,Grande São Paulo,True,motorbike,True,1,...,2022-08-01T00:00:00Z,1,0,0,0,0,0,62.255500,10.000000,1.0
1,1561210,Dennis Leonardo,Pereira Santos,M,1998-06-28,Grande São Paulo,True,motorbike,True,7,...,2022-08-01T00:00:00Z,6,1,0,0,0,0,43.444714,23.142857,3.0


In [22]:
df_churn_conc_by_city.columns

Index(['ID', 'FIRST_NAME', 'GENDER', 'CITY', 'SK.CREATED_AT::DATE',
       'TRANSPORT_MEDIA_TYPE', 'CARTAO', 'LEVEL_NAME', 'FECHA_ULT'],
      dtype='object')

In [23]:
filter = df_infos_gerais_conc_by_city["ID"].to_list()
mask = ~df_churn_conc_by_city["ID"].isin(filter)
df_churn_def = df_churn_conc_by_city[mask]
df_churn_def.head(2)

,ID,FIRST_NAME,GENDER,CITY,SK.CREATED_AT::DATE,TRANSPORT_MEDIA_TYPE,CARTAO,LEVEL_NAME,FECHA_ULT
1,1110698,Adriano Floriano Da Silva,M,Recife,2021-02-11,bicycle,True,bronze,2021-07-15T11:16:04Z
2,284886,Bruno,M,Grande São Paulo,2019-07-03,motorbike,False,bronze,2021-07-07T12:33:21Z


In [24]:
df_churn_def.rename(columns={
    "FIRST_NAME": "NOME", 
    "GENDER": "GENERO",
    "CITY": "CIDADE",
    "TRANSPORT_MEDIA_TYPE": "TRANSPORTE"
    }, inplace=True)

df_churn_def.drop(columns=["SK.CREATED_AT::DATE", "CARTAO", "FECHA_ULT", "LEVEL_NAME"], inplace=True)
df_churn_def["IS_ACTIVE"] = False

/usr/local/lib/python3.7/dist-packages/pandas/core/frame.py:5047: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  errors=errors,
/usr/local/lib/python3.7/dist-packages/pandas/core/frame.py:4913: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  errors=errors,
/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  if __name__ == '__

In [25]:
df_churn_def.head()

,ID,NOME,GENERO,CIDADE,TRANSPORTE,IS_ACTIVE
1,1110698,Adriano Floriano Da Silva,M,Recife,bicycle,False
2,284886,Bruno,M,Grande São Paulo,motorbike,False
9,172505,George,M,Grande São Paulo,motorbike,False
20,1030695,Leonan,M,Grande São Paulo,motorbike,False
31,934331,Wallace,M,Grande São Paulo,motorbike,False


In [26]:
df_consol = pd.concat([df_infos_gerais_conc_by_city, df_churn_def])
df_consol

,ID,NOME,SOBRENOME,GENERO,DATA_NASCIMENTO,CIDADE,IS_ACTIVE,TRANSPORTE,AUTO_ACEITE,COUNT_ORDERS_LAST_7D,...,ULTIMO_PEDIDO,COUNT_ORDERS_RESTAURANTES,COUNT_ORDERS_MERCADO,COUNT_ORDERS_FARMACIA,COUNT_ORDERS_EXPRESS,COUNT_ORDERS_ECOMMERCE,COUNT_ORDERS_ANTOJO,FRETE_MEDIO,COOKING_TIME_MEDIO,ITENS_MEDIO
0,1561246,Wilton Jhonne,Da Silva Abreu,M,1988-04-21,Grande São Paulo,True,motorbike,True,1.0,...,2022-08-01T00:00:00Z,1.0,0.0,0.0,0.0,0.0,0.0,62.255500,10.000000,1.00
1,1561210,Dennis Leonardo,Pereira Santos,M,1998-06-28,Grande São Paulo,True,motorbike,True,7.0,...,2022-08-01T00:00:00Z,6.0,1.0,0.0,0.0,0.0,0.0,43.444714,23.142857,3.00
3,1561173,Evellyn Dos,Santos Massal,F,1994-07-21,Grande São Paulo,True,motorbike,True,4.0,...,2022-08-01T00:00:00Z,4.0,0.0,0.0,0.0,0.0,0.0,47.236750,7.250000,1.25
5,1561061,Kouardo Silva,Gomes De Araujo,M,1993-07-17,Recife,True,motorbike,True,2.0,...,2022-08-01T00:00:00Z,1.0,0.0,0.0,0.0,0.0,0.0,35.555500,10.000000,2.50
6,1561019,Jessica Silva,Do Nascimento,F,2004-02-01,Fortaleza,True,bicycle,True,0.0,...,2022-08-01T00:00:00Z,0.0,0.0,0.0,0.0,0.0,0.0,57.805500,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
152216,1502887,Ricardo,NaN,M,NaN,Curitiba,False,motorbike,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
152217,1493340,Evandro,NaN,M,NaN,Belo Horizonte,False,motorbike,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
152220,1503585,Daniel,NaN,M,NaN,Curitiba,False,bicycle,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
152221,1399642,Fernando,NaN,M,NaN,Grande São Paulo,False,motorbike,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
type(df_consol["DATA_NASCIMENTO"].iloc[0])

str

In [27]:
df_consol["IS_ACTIVE"].value_counts(normalize=True)

Quasi    0.633805
True     0.245939
False    0.120256
Name: IS_ACTIVE, dtype: float64